# GRPO on GSM8K: Colab quick run

Runs a short GRPO training (LoRA, Qwen2.5-0.5B-Instruct) on a free T4 and compares it with the zero-shot model. Runtime → Change runtime type → **T4 GPU**.

In [ ]:
!git clone https://github.com/<your-username>/grpo-gsm8k.git
%cd grpo-gsm8k
!pip install -q -r requirements.txt
!pip install -q -e .

T4 has no bf16 support, so we switch to fp16 and shorten everything.

In [ ]:
COLAB = ['grpo.bf16=false', 'grpo.fp16=true', 'grpo.max_steps=100',
         'grpo.num_generations=4', 'grpo.per_device_train_batch_size=4',
         'grpo.max_completion_length=200', 'data.train_samples=500']
!python scripts/train_grpo.py --config configs/base.yaml output_dir=outputs/colab {' '.join(COLAB)}

In [ ]:
!python scripts/evaluate.py --model Qwen/Qwen2.5-0.5B-Instruct --num-samples 200 --out results/zero_shot
!python scripts/evaluate.py --model Qwen/Qwen2.5-0.5B-Instruct --adapter outputs/colab --num-samples 200 --out results/colab

In [ ]:
!python scripts/plot.py --runs colab
from IPython.display import Image, Markdown, display
display(Markdown(open('figures/results.md').read()))
display(Image('figures/training_curves.png'))

In [ ]:
import json
for line in list(open('results/colab/samples.jsonl'))[:3]:
    r = json.loads(line)
    print('gold:', r['gold'], '| pred:', r['prediction'], '| correct:', r['correct'])
    print(r['completion'][:500], '\n' + '-' * 80)